In [2]:
import undetected_chromedriver as uc
from selenium.webdriver.common.by import By
import pandas as pd
import time
import random
from bs4 import BeautifulSoup
import os
# 1. 드라이버 설정 (버전 146 고정 시도)
options = uc.ChromeOptions()
options.add_argument('--start-maximized')
options.add_argument('--disable-popup-blocking')
try:
    # 지정하신 146 버전으로 실행 시도
    driver = uc.Chrome(options=options, version_main=146)
except Exception as e:
    print(f"지정 버전 실행 실패로 기본 설정 실행: {e}")
    driver = uc.Chrome(options=options)
# 2. 타겟 게시물 링크
target_url = "https://www.reddit.com/r/politics/comments/fdx4c4/megathread_elizabeth_warren_to_suspend/"
try:
    driver.get(target_url)
    print(f"게시물 접속 중: {target_url}")
    
    # 레딧은 첫 접속 시 캡차가 뜰 수 있으니 확인 시간을 가집니다.
    print("\n[대기] 캡차가 뜨면 해결해주세요. 댓글이 보이기 시작하면 엔터를 누르세요.")
    input("로딩 확인 후 엔터 >>> ")
    # 3. '댓글 더보기' 무한 반복 클릭 로직
    print("--- 댓글 펼치기 작업을 시작합니다 (시간을 넉넉히 가집니다) ---")
    
    click_count = 0
    fail_count = 0 # 연속 실패 횟수 체크
    while True:
        # 자바스크립트: 버튼을 찾아 화면 중앙으로 옮기고, '진짜' 클릭 이벤트를 발생시킵니다.
        js_click_script = """
        let buttons = document.querySelectorAll('button');
        let found = false;
        for (let btn of buttons) {
            let txt = btn.innerText.toLowerCase();
            // 'more', 'replies', 'view more' 등 키워드 매칭
            if (txt.includes('more') || txt.includes('replies') || txt.includes('view')) {
                btn.scrollIntoView({block: 'center', behavior: 'smooth'});
                
                // 일반 click() 보다 확실한 이벤트 발송
                btn.dispatchEvent(new MouseEvent('click', {bubbles: true, cancelable: true, view: window}));
                found = true;
                break; 
            }
        }
        return found;
        """
        
        # 페이지 로딩 유도를 위한 미세 스크롤
        driver.execute_script("window.scrollBy(0, 300);")
        time.sleep(1.5)
        
        # 버튼 클릭 실행
        is_clicked = driver.execute_script(js_click_script)
        
        if is_clicked:
            click_count += 1
            fail_count = 0 # 성공했으니 실패 카운트 초기화
            
            # --- 핵심: 대기 시간 대폭 증가 (5초 ~ 10초 사이 랜덤) ---
            wait_time = random.uniform(5.0, 10.0)
            print(f"   - [{click_count}회] 클릭 성공! 새 댓글 로딩 대기 ({wait_time:.1f}초)...")
            time.sleep(wait_time)
            
            # 너무 많이 클릭했을 경우 잠시 긴 휴식 (서버 차단 방지)
            if click_count % 15 == 0:
                print("   [휴식] 안정적인 수집을 위해 15초간 추가 대기합니다...")
                time.sleep(15)
        else:
            # 버튼이 안 보일 경우, 바닥까지 내려서 재확인
            fail_count += 1
            print(f"   - 버튼을 찾을 수 없음 (시도 {fail_count}/3)... 스크롤 중")
            driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
            time.sleep(4)
            
            # 3번 연속으로 버튼을 못 찾으면 모든 댓글이 펼쳐진 것으로 간주
            if fail_count >= 3:
                print("--- 모든 댓글이 펼쳐졌거나 더 이상 로드할 데이터가 없습니다. ---")
                break
    # 4. 데이터 파싱 (BeautifulSoup)
    print("--- 데이터 추출 시작 ---")
    soup = BeautifulSoup(driver.page_source, 'html.parser')
    comment_elements = soup.select('shreddit-comment')
    
    comment_final = []
    for comm in comment_elements:
        content_div = comm.select_one('div[slot="comment"]')
        if content_div:
            # 줄바꿈 제거 및 앞뒤 공백 정리
            text = content_div.get_text(strip=True).replace('\n', ' ').strip()
            if text:
                comment_final.append(text)
    # 5. 저장 경로 설정 및 저장
    # 폴더가 없을 경우를 대비해 생성 로직 추가
    save_path = "../data/raw/"
    if not os.path.exists(save_path):
        os.makedirs(save_path)
        
    df = pd.DataFrame({"Comment": comment_final})
    file_name = "reddit_single_post_result.csv"
    full_file_path = os.path.join(save_path, file_name)
    
    df.to_csv(full_file_path, encoding="utf-8-sig", index=False)
    
    print("-" * 30)
    print(f"수집 완료! 총 {len(comment_final)}개의 댓글을 저장했습니다.")
    print(f"저장 위치: {full_file_path}")
except Exception as e:
    print(f"에러 발생: {e}")
finally:
    print("3초 후 브라우저를 종료합니다.")
    time.sleep(3)
    driver.quit()

게시물 접속 중: https://www.reddit.com/r/politics/comments/fdx4c4/megathread_elizabeth_warren_to_suspend/

[대기] 캡차가 뜨면 해결해주세요. 댓글이 보이기 시작하면 엔터를 누르세요.
--- 댓글 펼치기 작업을 시작합니다 (시간을 넉넉히 가집니다) ---
   - 버튼을 찾을 수 없음 (시도 1/3)... 스크롤 중
   - 버튼을 찾을 수 없음 (시도 2/3)... 스크롤 중
   - 버튼을 찾을 수 없음 (시도 3/3)... 스크롤 중
--- 모든 댓글이 펼쳐졌거나 더 이상 로드할 데이터가 없습니다. ---
--- 데이터 추출 시작 ---
------------------------------
수집 완료! 총 1465개의 댓글을 저장했습니다.
저장 위치: ../data/raw/reddit_single_post_result.csv
3초 후 브라우저를 종료합니다.


> API KEY 승인 대기중